# Training data

In [1]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [2]:
""" 
IMPORTS
"""
import os
import numpy as np
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --Machine learning and statistics

import pickle
from one.api import ONE
import concurrent.futures

# Get my functions
from functions import create_grouped_gradient_palette
# Get my functions
functions_path = prefix + '/representation_learning_variability/Functions/'
os.chdir(functions_path)
from one_functions_generic import download_subjectTables

one = ONE(mode='remote')

In [3]:
# Get my functions
def idxs_from_files(design_matrices):
    
    idxs = []
    mouse_names = []
    for m, mat in enumerate(design_matrices):
        mouse_name = design_matrices[m][51:]
        eid = design_matrices[m][14:50]
        idx = str(eid + '_' + mouse_name)

        if len(idxs) == 0:
            idxs = idx
            mouse_names = mouse_name
        else:
            idxs = np.hstack((idxs, idx))
            mouse_names = np.hstack((mouse_names, mouse_name))
            
    return idxs, mouse_names


""" 
LOAD DATA AND PARAMETERS
"""
# LOAD DATA

data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/'
all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

learning_data_path =   prefix + 'representation_learning_variability/paper-individuality/data/training_data/'


In [4]:
all_files = os.listdir(learning_data_path)
mice_to_download = []

for m, mouse_name in enumerate(np.unique(mouse_names)):
    
    filename = 'training_data_trials_'+mouse_name
    if filename not in all_files:
        mice_to_download.append(mouse_name)
    
print(len(mice_to_download))

32


# Download and save data

In [23]:
    subject=mouse_name
    trials=True
    training=True
    target_path=None
    tag=None
    overwrite=False
    check_updates=True
                           
    if target_path is None:
        target_path = Path(one.cache_dir).joinpath('aggregates')
        target_path.mkdir(exist_ok=True)
    else:
        assert target_path.exists(), 'The target_path you passed does not exist.'

    # Get the datasets
    trials_ds = []
    training_ds = []
    if subject:
        try:
            subject_id = uuid.UUID(subject)
        except ValueError:
            subject_id = one.alyx.rest('subjects', 'list', nickname=subject)[0]['id']
        if trials:
            trials_ds.extend(one.alyx.rest('datasets', 'list', name='_ibl_subjectTrials.table.pqt',
                                           django=f'object_id,{subject_id}'))
        if training:
            training_ds.extend(one.alyx.rest('datasets', 'list', name='_ibl_subjectTraining.table.pqt',
                                             django=f'object_id,{subject_id}'))
    else:
        if tag:
            if trials:
                trials_ds.extend(one.alyx.rest('datasets', 'list', name='_ibl_subjectTrials.table.pqt', tag=tag))
            if training:
                training_ds.extend(one.alyx.rest('datasets', 'list', name='_ibl_subjectTraining.table.pqt', tag=tag))
        else:
            if trials:
                trials_ds.extend(one.alyx.rest('datasets', 'list', name='_ibl_subjectTrials.table.pqt'))
            if training:
                training_ds.extend(one.alyx.rest('datasets', 'list', name='_ibl_subjectTraining.table.pqt'))

    # Set up the bucket
    s3, bucket_name = aws.get_s3_from_alyx(alyx=one.alyx)

    all_out = []
    for ds_list in [trials_ds, training_ds]:
        out_paths = []
        for ds in ds_list:
            # Check if file_records exists and is not empty
            if not ds.get('file_records') or len(ds['file_records']) == 0:
                print(f"Warning: No file records found for dataset {ds.get('url', 'unknown')}")
                continue
            relative_path = add_uuid_string(ds['file_records'][0]['relative_path'], ds['url'][-36:])
            src_path = 'aggregates/' + str(relative_path)
            dst_path = target_path.joinpath(relative_path)
            if check_updates:
                out = aws.s3_download_file(src_path, dst_path, s3=s3, bucket_name=bucket_name, overwrite=overwrite)
            else:
                out = dst_path

            if out and out.exists():
                out_paths.append(out)
            else:
                print(f'Downloading of {src_path} table failed.')
        all_out.append(out_paths)

File aggregates/Subjects/churchlandlab/CSHL045/#2025-03-03#/_ibl_subjectTrials.table.dd602a2e-97dd-4c42-ac57-c14a57c6c87d.pqt not found on ibl-brain-wide-map-private


In [10]:
for m, mouse_name in enumerate(mice_to_download):
    # Trials data
    print(mouse_name)
    files = download_subjectTables(one, mouse_name, trials=True, training=True,
                        target_path=None, tag=None, overwrite=False, check_updates=True)
    trials, training = [pd.read_parquet(file) for file in files]
    training = training.reset_index()
    trained_date = pd.to_datetime(training.loc[training['training_status'].isin(['trained 1a', 'trained 1b']), 'date']).dt.date
    
    sessions = trials[['task_protocol', 'session_start_time', 'session']].drop_duplicates()
    training_sessions = sessions.loc[sessions['task_protocol'].str.contains('training')]
    trials['session_date'] = pd.to_datetime(trials['session_start_time']).dt.date
    mouse_data = trials.loc[trials['session_date']<np.min(trained_date)]

    # Save data
    mouse_data.to_parquet(learning_data_path+'training_data_trials_'+mouse_name,compression='gzip') 
    print('Successful for mouse ' + mouse_name)
    print(len(mouse_data))


CSHL045


File aggregates/Subjects/churchlandlab/CSHL045/#2025-03-03#/_ibl_subjectTrials.table.dd602a2e-97dd-4c42-ac57-c14a57c6c87d.pqt not found on ibl-brain-wide-map-private


Successful for mouse CSHL045
61198
CSHL047


File aggregates/Subjects/churchlandlab/CSHL047/#2025-03-03#/_ibl_subjectTrials.table.2edc7114-e8a2-4ad6-afc6-ad5d0f13e340.pqt not found on ibl-brain-wide-map-private


Successful for mouse CSHL047
36954
CSHL049


...

Successful for mouse DY_011
3076
DY_013


File aggregates/Subjects/danlab/DY_013/#2025-03-03#/_ibl_subjectTrials.table.9760c656-d994-47cf-b115-e55d7678e534.pqt not found on ibl-brain-wide-map-private


Successful for mouse DY_013
8350
DY_014


File aggregates/Subjects/danlab/DY_014/#2025-03-03#/_ibl_subjectTrials.table.a444fc55-8d64-461d-8c15-6f3415cddba7.pqt not found on ibl-brain-wide-map-private


Successful for mouse DY_014
10230
NR_0028
Successful for mouse NR_0028
14564
NYU-11
Successful for mouse NYU-11
13173
NYU-12
Successful for mouse NYU-12
14582
SWC_038
Successful for mouse SWC_038
26092
SWC_042
Successful for mouse SWC_042
15820
ZFM-01576
Successful for mouse ZFM-01576
41736
ZFM-01577
Successful for mouse ZFM-01577
7600
ZFM-01592
Successful for mouse ZFM-01592
7474
ZM_1898
Successful for mouse ZM_1898
64470
ZM_2240
Successful for mouse ZM_2240
9944
ZM_2241
Successful for mouse ZM_2241
31540
ZM_2245
Successful for mouse ZM_2245
23936
ZM_3003
Successful for mouse ZM_3003
13848
ibl_witten_13
Successful for mouse ibl_witten_13
13866
